In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = os.path.dirname(os.getcwd())
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import json

with open("./data/flickr30k/subset/train/data.json") as f:
    raw = json.load(f)

data = []
for item in raw:
    for cap in item["captions"]:
        data.append({
            "image": item["img"],
            "text": cap.lower()
        })

print(len(data))
if data:
    print("第一条数据示例:", data[0])

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class FlickrDataset(Dataset):
    def __init__(self, data, processor):
        self.data = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item["image"]).convert("RGB")

        prompt = "Question: Describe this image. Answer: "
        text = prompt + item["text"]

        inputs = self.processor(
            images=image,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=64
        )

        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = inputs["input_ids"]
        return inputs

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

model_name = "Salesforce/blip2-flan-t5-xl"
cache_dir = "./model/blip/blip2_cache"
dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# processor
processor = Blip2Processor.from_pretrained(
    model_name,
    cache_dir=cache_dir
    )
# model
model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    device_map="auto" if device.type == "cuda" else None
    )

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q", "v",                 # T5 attention
        "query", "key", "value"   # Q-Former
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./blip/blip2_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=20,
    save_steps=500,
    report_to="none"
)

In [ ]:
from transformers import Trainer

train_dataset = FlickrDataset(data, processor)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
model.save_pretrained("./blip/blip2_lora")
processor.save_pretrained("./blip/blip2_lora")

In [ ]:
# 手动清理
import torch
import gc

# 删除训练器释放引用
del trainer
del model

# 清空CUDA缓存
torch.cuda.empty_cache()

# 触发Python垃圾回收
gc.collect()

# 查看显存使用情况
print(f"显存分配: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"显存缓存: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

In [ ]:
from peft import PeftModel

model_name = "Salesforce/blip2-flan-t5-xl"
cache_dir = "./model/blip/blip2_cache"
dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model
base = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    device_map="auto" if device.type == "cuda" else None
    )


model = PeftModel.from_pretrained(base, "./blip/blip2_lora")

In [ ]:
quit()